# Resume / Candidate Screening System

This notebook demonstrates resume cleaning, skill extraction, scoring, and candidate ranking.

In [ ]:
from pathlib import Path
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
ROOT = Path('.')
job_path = ROOT / 'data' / 'job_descriptions' / 'software_engineer.txt'
resumes_dir = ROOT / 'data' / 'resumes'

job_description = job_path.read_text(encoding='utf-8', errors='ignore')
resumes = {p.stem: p.read_text(encoding='utf-8', errors='ignore') for p in sorted(resumes_dir.glob('*.txt'))}

len(resumes), list(resumes.keys())

In [ ]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^a-z0-9+#.\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

required_skills = {
    'python', 'sql', 'aws', 'docker', 'kubernetes',
    'machine learning', 'nlp', 'communication', 'api', 'git'
}

def extract_skills(text: str, catalog: set[str]) -> set[str]:
    normalized = clean_text(text)
    found = set()
    for skill in catalog:
        pattern = rf'\b{re.escape(clean_text(skill))}\b'
        if re.search(pattern, normalized):
            found.add(skill)
    if 'apis' in normalized and 'api' in catalog:
        found.add('api')
    return found

In [ ]:
def similarity(job: str, resume: str) -> float:
    docs = [clean_text(job), clean_text(resume)]
    vec = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
    mat = vec.fit_transform(docs)
    return float(cosine_similarity(mat[0:1], mat[1:2])[0][0])

rows = []
for candidate, text in resumes.items():
    sim = similarity(job_description, text)
    found = extract_skills(text, required_skills)
    matched = sorted(found.intersection(required_skills))
    missing = sorted(required_skills.difference(found))
    skill_match = len(matched) / len(required_skills)
    final_score = 0.7 * sim + 0.3 * skill_match
    rows.append({
        'candidate': candidate,
        'final_score': round(final_score, 4),
        'text_similarity': round(sim, 4),
        'skill_match': round(skill_match, 4),
        'matched_skills': ', '.join(matched),
        'missing_skills': ', '.join(missing),
    })

ranking_df = pd.DataFrame(rows).sort_values('final_score', ascending=False).reset_index(drop=True)
ranking_df.index = ranking_df.index + 1
ranking_df

## Conclusion

The top-ranked candidate has the best combined profile from semantic text relevance and required skill coverage.
This makes the ranking explainable for recruiters using both score components and missing skill gaps.